# Notebook 00 - Sonar intuition and baseline parameters

## What this notebook teaches
- what active sonar means in practice: ping, then listen
- why underwater sensing is built around sound speed in water
- how duty cycle, round-trip delay, and sample delay follow from the baseline scene

## Where we are in the story
An active sonar measures range by timing how long sound takes to travel through water, reflect from a target, and come back. Before building sweeps, matched filters, or Doppler maps, we need one clean baseline scene and a feel for the transmit-listen rhythm that makes active sonar possible.

In [ ]:
from helpers import BASELINE_SCENE

sound_speed_m_s = BASELINE_SCENE["sound_speed_m_s"]
ping_width_s = BASELINE_SCENE["ping_width_s"]
inter_ping_interval_s = BASELINE_SCENE["inter_ping_interval_s"]
sample_rate_hz = BASELINE_SCENE["sample_rate_hz"]
target_range_m = BASELINE_SCENE["target_range_m"]

print(f"Sound speed: {sound_speed_m_s:.0f} m/s")
print(f"Ping width: {ping_width_s:.3f} s")
print(f"Inter-ping interval: {inter_ping_interval_s:.3f} s")
print(f"Sample rate: {sample_rate_hz:.0f} Hz")
print(f"Target range: {target_range_m:.1f} m")

## Baseline values and governing equations
The first two equations in the whole track are:

$D = \tau / T$

for duty cycle, and

$R = c \cdot \tau_{delay} / 2$

for range from round-trip delay. Here $\tau$ is the ping width, $T$ is the time from one ping start to the next, $c$ is sound speed in water, and $\tau_{delay}$ is the echo delay.

## Teach it directly first
Before using any helper function, compute the key quantities by hand so the arithmetic stays visible. The whole point is to make the transmit-listen rhythm feel physical rather than abstract.

In [ ]:
duty_cycle_value = ping_width_s / inter_ping_interval_s
listen_window_s = inter_ping_interval_s - ping_width_s

print(f"Duty cycle D = tau / T = {ping_width_s:.3f} / {inter_ping_interval_s:.3f} = {duty_cycle_value:.2f}")
print(f"Quiet listening time each ping: {listen_window_s:.3f} s")

In [ ]:
round_trip_delay_s = 2.0 * target_range_m / sound_speed_m_s
delay_samples = round(round_trip_delay_s * sample_rate_hz)
recovered_range_m = sound_speed_m_s * round_trip_delay_s / 2.0

print(f"Round-trip delay = 2R/c = 2 * {target_range_m:.1f} / {sound_speed_m_s:.0f} = {round_trip_delay_s:.3f} s")
print(f"Delay in samples = {round_trip_delay_s:.3f} * {sample_rate_hz:.0f} = {delay_samples} samples")
print(f"Recovered range = c * tau_delay / 2 = {recovered_range_m:.1f} m")

## Repeat the same idea with helpers
Now that the arithmetic is visible, the helpers in this same notebooks folder can package the same operations cleanly for later notebooks. They should never replace the direct explanation; they should only repeat it.

In [ ]:
import matplotlib.pyplot as plt

from helpers import delay_samples_for_range, duty_cycle, range_from_delay

helper_duty_cycle = duty_cycle(ping_width_s, inter_ping_interval_s)
helper_delay_samples = delay_samples_for_range(sound_speed_m_s, target_range_m, sample_rate_hz)
helper_range_m = range_from_delay(sound_speed_m_s, round_trip_delay_s)

print(f"Helper duty cycle: {helper_duty_cycle:.2f}")
print(f"Helper delay samples: {helper_delay_samples}")
print(f"Helper recovered range: {helper_range_m:.1f} m")

fig, ax = plt.subplots(figsize=(8, 2.5))
ax.axvspan(0.0, ping_width_s, color="tab:blue", alpha=0.7, label="Transmit ping")
ax.axvspan(ping_width_s, inter_ping_interval_s, color="tab:green", alpha=0.25, label="Listen window")
ax.axvline(round_trip_delay_s, color="tab:red", linestyle="--", label="Echo arrival")
ax.set_xlim(0.0, inter_ping_interval_s)
ax.set_yticks([])
ax.set_xlabel("Time within one ping cycle (s)")
ax.set_title("Transmit first, then listen for the echo")
ax.legend(loc="upper right")
plt.show()

## Checkpoint
- Why does the range equation divide by 2 instead of 1?
- What happens to the listen window if the ping width grows while the inter-ping interval stays fixed?
- Why is 0.1 s a reasonable round-trip delay in water but not in radar?

## Common mistake
Confusing one-way travel time with round-trip delay. The measured echo delay includes the trip out to the target and the trip back.

## Stretch
Keep the same sample rate and sound speed, but move the target to 120 m. Compute the new delay in seconds and samples.

## Summary
Active sonar works by transmitting briefly and listening for a much longer period. The first-principles quantities in that rhythm are duty cycle, round-trip delay, and sample delay, and all of them come directly from sound speed in water and the chosen baseline geometry.